In [2]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Processing function ===
def calculate_maximum_6month_mean_no2(start_year, end_year, monthly_mda8, mda8_no2):
    years = list(range(start_year, end_year + 1))

    max_vals = []

    for year in years:
        # Define window: Jan of this year to Mar of next year
        start = f"{year}-01"
        end = f"{year + 1}-03"

        # Subset to this window
        subset = monthly_mda8.sel(time=slice(start, end))

        # Compute 6-month rolling mean along time
        rolling_6m = subset.rolling(time=6, center=False).mean()

        # Max 6-month O3 and index
        idx_of_max = rolling_6m.argmax(dim="time")  # (lat, lon)

        # Now build indices for 6-month window
        idx_window = xr.DataArray(
            np.arange(-5, 1),
            dims=["window"],
        ) + idx_of_max.expand_dims(window=6)

        # Mask invalid indices
        valid = (idx_window >= 0) & (idx_window < subset.time.size)
        idx_window = idx_window.where(valid, 0)

        # Use isel smartly
        no2_selected = mda8_no2.isel(time=idx_window)  # shape (window, lat, lon)

        # Now average over the 8-hour window
        no2_6m_mean = no2_selected.where(valid).mean(dim="window", skipna=True)

        # Expand dimensions for consistent output
        max_val = no2_6m_mean.expand_dims(year=[year])

        max_vals.append(max_val)

    # Combine across years
    annual_max_6m_no2 = xr.concat(max_vals, dim="year")

    return annual_max_6m_no2

In [5]:
# === Path config ===
O3_DIR = "/glade/work/awells/air_quality/CESM/MDA8/"
NO2_DIR = "/glade/work/awells/air_quality/CESM/NO2_MDA8/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/NO2_6m/"
# SCENARIOS = ["ARISE", "SSP245"]
SCENARIOS = ["SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(10, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "20350101-20691231"
        else:
            dates = "20200101-20691231"
        file_list_o3 = [f"{O3_DIR}MDA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"]
        file_list_no2 = [f"{NO2_DIR}MDA8_NO2_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"]
        NO2_6m = []

        for f_no2, f_o3 in zip(file_list_no2, file_list_o3):
            for f in [f_no2, f_o3]:
                if not os.path.exists(f):
                    raise ValueError(f"Missing: {f}")

            print(f"Reading {os.path.basename(f_no2)}")
            monthly_mda8 = xr.open_dataarray(f_o3)
            mda8_no2 = xr.open_dataarray(f_no2)

            # Create list of years to calculate over
            start_year = int(str(monthly_mda8.time.dt.year[0].values))
            end_year = int(str(monthly_mda8.time.dt.year[-1].values))  # final year will be 12 months rather than 15

            annual_max_6m_no2 = calculate_maximum_6month_mean_no2(start_year, end_year, monthly_mda8, mda8_no2)

            NO2_6m.append(annual_max_6m_no2)

        if NO2_6m:
            combined = xr.concat(NO2_6m, dim="year")

            out_file = f"NO2_6m_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            combined.to_netcdf(out_path)

print("All processing complete.")

Processing SSP245, Ensemble 10
Reading MDA8_NO2_CESM2_SSP245_10_20200101-20691231.nc
Saving to /glade/work/awells/air_quality/CESM/NO2_6m/NO2_6m_CESM2_SSP245_10_20200101-20691231.nc
All processing complete.
